# Playwright MCP – Minimal Scraping Test
Verify the `@playwright/mcp` server works in WSL by scraping one player page from basketball-reference.com.

In [1]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner
from agents.mcp import MCPServerStdio
from agents import OpenAIChatCompletionsModel
from openai import AsyncOpenAI

load_dotenv()

True

In [2]:
MCP_ARGS = [
    "-y", "@playwright/mcp@latest",
    "--no-sandbox",
    "--executable-path", "/home/raamk/.cache/ms-playwright/chromium-1212/chrome-linux64/chrome",
]

server = MCPServerStdio(
    name="Playwright Browser",
    params={"command": "npx", "args": MCP_ARGS},
)

In [3]:
google_api_key = os.getenv("GOOGLE_API_KEY")
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

model_name = "gemini-3.1-flash-lite-preview"
model = OpenAIChatCompletionsModel(model=model_name, openai_client=gemini_client)

In [ ]:
ADV_STATS_PROMPT = """
You are a web scraper agent with access to a Playwright browser.

Instructions:

1. Given a player's full name as a string input, construct the Basketball Reference search URL:
   https://www.basketball-reference.com/search/search.fcgi?search=<player name>

2. Use browser_navigate to open that URL.

3. From the search results page, find the first link whose URL contains "/players/" and click it.

4. On the player's page, locate the "Advanced" stats table (id="advanced").

5. Extract all season rows, excluding any career totals rows.

6. For each season, extract:
   - year (the Season label, e.g. "2004-05")
   - PER
   - TS% (as TS_pct)
   - WS
   - WS/48 (as WS_per_48)
   - BPM
   - VORP

7. Return the data using the structured output schema provided.
"""


In [ ]:
import pandas as pd
from pydantic import BaseModel


class SeasonAdvancedStats(BaseModel):
    """Advanced stats for a single season."""
    year: str       # e.g. "2004-05"
    PER: float
    TS_pct: float   # TS%
    WS: float
    WS_per_48: float  # WS/48
    BPM: float
    VORP: float


class PlayerAdvancedStats(BaseModel):
    """All seasons of advanced stats for a player."""
    seasons: list[SeasonAdvancedStats]


def create_bbref_agent(player_name: str):
    return Agent(
        name="BBRef Scraper",
        instructions=f"{ADV_STATS_PROMPT}\nPlayer name: {player_name}",
        mcp_servers=[server],
        model=model,
        output_type=PlayerAdvancedStats,
    )


async def scrape_player(player_name: str) -> pd.DataFrame:
    """Scrape advanced stats and return a DataFrame with columns:
    Year, PER, TS%, WS, WS/48, BPM, VORP.
    """
    agent = create_bbref_agent(player_name)
    async with server:
        result = await Runner.run(agent, input=f"Extract advanced stats for {player_name}", max_turns=25)
    stats: PlayerAdvancedStats = result.final_output
    rows = [
        {
            "Year": s.year,
            "PER": s.PER,
            "TS%": s.TS_pct,
            "WS": s.WS,
            "WS/48": s.WS_per_48,
            "BPM": s.BPM,
            "VORP": s.VORP,
        }
        for s in stats.seasons
    ]
    return pd.DataFrame(rows)

In [ ]:
player_name = "Steve Nash"
df = await scrape_player(player_name)
df

## Stat Leaders by Year
Scrape the yearly leader (top score) for BPM, WS/48, PER, and VORP from basketball-reference.com.

In [ ]:
STAT_LEADER_PAGES = {
    "BPM": "https://www.basketball-reference.com/leaders/bpm_yearly.html",
    "WS/48": "https://www.basketball-reference.com/leaders/ws48_yearly.html",
    "PER": "https://www.basketball-reference.com/leaders/per_yearly.html",
    "VORP": "https://www.basketball-reference.com/leaders/vorp_yearly.html",
}

LEADERS_PROMPT = """
You are a web scraper agent with access to a Playwright browser.

Instructions:

1. Navigate to the given URL using browser_navigate.

2. The page contains a year-by-year leaders table. Each row has:
   - A season/year (e.g. "2024-25" or a link to the season)
   - The league (NBA/ABA)
   - The leader's name
   - Their team
   - The stat value (a number)

3. Extract EVERY row from the table. For each row, capture:
   - year: the season string, e.g. "2024-25"
   - value: the leading stat value as a float

4. Only include NBA rows (skip ABA rows if present).

5. Return the data using the structured output schema provided.
"""


class YearlyLeaderEntry(BaseModel):
    """A single year's leading stat value."""
    year: str     # e.g. "2024-25"
    value: float  # the leading stat score


class YearlyLeaderTable(BaseModel):
    """All yearly leader entries for one stat."""
    entries: list[YearlyLeaderEntry]


async def scrape_stat_leaders(stat_name: str, url: str) -> pd.DataFrame:
    """Scrape a single stat leader page and return a DataFrame with Year and the stat column."""
    agent = Agent(
        name=f"{stat_name} Leaders Scraper",
        instructions=f"{LEADERS_PROMPT}\nURL to scrape: {url}",
        mcp_servers=[server],
        model=model,
        output_type=YearlyLeaderTable,
    )
    async with server:
        result = await Runner.run(agent, input=f"Extract yearly leaders from {url}", max_turns=25)
    table: YearlyLeaderTable = result.final_output
    df = pd.DataFrame([{"Year": e.year, stat_name: e.value} for e in table.entries])
    return df


async def scrape_all_stat_leaders() -> pd.DataFrame:
    """Scrape all stat leader pages and merge into a single DataFrame."""
    dfs = []
    for stat_name, url in STAT_LEADER_PAGES.items():
        print(f"Scraping {stat_name} leaders...")
        df = await scrape_stat_leaders(stat_name, url)
        print(f"  Got {len(df)} rows")
        dfs.append(df)
    # Merge all DataFrames on Year
    merged = dfs[0]
    for df in dfs[1:]:
        merged = merged.merge(df, on="Year", how="outer")
    merged = merged.sort_values("Year", ascending=False).reset_index(drop=True)
    return merged

In [ ]:
leaders_df = await scrape_all_stat_leaders()
leaders_df